# Notebook 2: Diagnostics

Before running any regressions, I need to check that the data has the statistical properties I'm assuming. This notebook handles three things:

1. Descriptive statistics for all five series, to document the basic empirical features of the data.
2. Augmented Dickey-Fuller stationarity tests, which OLS requires.
3. The contemporaneous correlation matrix, which gives a first read on how the series move together before I look at any predictive relationship.

Outputs of this notebook are three tables and two figures, all of which go into Section 4.1 of the paper.

** employ GARCH!!! **

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from pathlib import Path

Path("../figures").mkdir(exist_ok=True)

# load the main dataset from Day 2
df = pd.read_parquet("../data/regression_data.parquet")
cee_returns = pd.read_parquet("../data/cee_returns.parquet")
vix = pd.read_parquet("../data/vix.parquet")

print(f"Regression dataset: {df.shape}")
print(f"CEE returns:        {cee_returns.shape}")
print(f"VIX series:         {vix.shape}")
print(f"\nDate range: {df.index[0].date()} to {df.index[-1].date()}")
df.head()

Regression dataset: (1635, 6)
CEE returns:        (1636, 4)
VIX series:         (1637, 1)

Date range: 2018-01-04 to 2024-12-23


,VIX_level,VIX_change,WIG20_next,BUX_next,PX_next,ATX_next
Date,,,,,,
2018-01-04,9.22,0.070001,-0.001430,0.003636,0.000172,0.000124
2018-01-05,9.22,0.000000,0.007507,0.002718,0.003252,-0.000183
2018-01-08,9.52,0.300000,-0.007969,-0.005864,-0.004012,0.002336
2018-01-09,10.08,0.559999,-0.006790,-0.005491,-0.002493,0.009308
2018-01-10,9.82,-0.260000,0.009231,0.003060,0.005386,0.002765


## Load Data and Compute Descriptive Statistics

The descriptives table shows the standard summary statistics for the VIX (in index points) and the four CEE return series in percent per day.

I expect to see the typical features of equity returns: near-zero means, daily standard deviations in the 1 to 2 percent range, negative skewness (crashes deeper than rallies), and high excess kurtosis (fat tails). For the VIX, I expect a long-run mean in the high teens to low twenties and a maximum value far above the mean, corresponding to the March 2020 spike at 82.69.

In [ ]:
# build a clean descriptives table for all 5 series:

desc_data = pd.concat([
    vix["VIX"].rename("VIX"),
    cee_returns
], axis=1).dropna()

# convert CEE returns to percent for readability
desc_pct = desc_data.copy()
for col in ["WIG20", "BUX", "PX", "ATX"]:
    desc_pct[col] = desc_pct[col] * 100  # log returns × 100

# build the table
descriptives = pd.DataFrame({
    "Mean"    : desc_pct.mean(),
    "Median"  : desc_pct.median(),
    "Std Dev" : desc_pct.std(),
    "Min"     : desc_pct.min(),
    "Max"     : desc_pct.max(),
    "Skewness": desc_pct.apply(stats.skew),
    "Kurtosis": desc_pct.apply(stats.kurtosis),  # excess kurtosis
    "N"       : desc_pct.count()
}).round(3)

print("Table 1: Descriptive Statistics")
print("(VIX in index points; CEE returns in % per day)\n")
print(descriptives)

# save for the paper
descriptives.to_csv("../data/table_1_descriptives.csv")

Table 1: Descriptive Statistics
(VIX in index points; CEE returns in % per day)

         Mean  Median  Std Dev     Min     Max  Skewness  Kurtosis     N
VIX    19.910  18.120    7.789   9.220  82.690     2.676    12.742  1636
WIG20  -0.007  -0.017    1.508 -14.246   8.099    -0.707     8.347  1636
BUX     0.042   0.099    1.337 -12.268   6.003    -1.382    12.386  1636
PX      0.029   0.067    0.980  -8.160   7.369    -1.040    11.891  1636
ATX     0.003   0.072    1.416 -14.712  10.226    -1.162    15.360  1636

Saved → data/table_1_descriptives.csv


## Stationarity: Augmented Dickey-Fuller Tests

Standard OLS inference requires the variables to be stationary. If any of the regression variables had a unit root, the t-statistics in Section 4 would be unreliable.

I run the ADF test on all six series I'll use in the regressions: the VIX level, the VIX change, and the four CEE return series. The null hypothesis is non-stationarity (a unit root is present); rejecting the null at conventional levels means the series is stationary.

For returns, stationarity is guaranteed by the way returns are constructed. For the VIX level it's less obvious. 

But the literature (notably Fernandes, Medeiros, and Scharth 2014) shows that VIX is mean-reverting around its long-run average over multi-year samples, which is exactly what stationarity requires.

In [ ]:
# Augmented Dickey-Fuller test: null hypothesis is that the series has a unit root (non-stationary)
# rejecting null (p < 0.05) means the series IS stationary (what we want for OLS)

def adf_test(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    return {
        "Series"         : name,
        "ADF Statistic"  : round(result[0], 4),
        "p-value"        : round(result[1], 4),
        "Lags Used"      : result[2],
        "N Obs"          : result[3],
        "Critical 1%"    : round(result[4]["1%"], 3),
        "Critical 5%"    : round(result[4]["5%"], 3),
        "Stationary?"    : "Yes" if result[1] < 0.05 else "No"
    }

# test all five series we use in regression
tests = []
tests.append(adf_test(vix["VIX"],           "VIX (level)"))
tests.append(adf_test(vix["VIX"].diff(),    "VIX (change)"))
for col in ["WIG20", "BUX", "PX", "ATX"]:
    tests.append(adf_test(cee_returns[col], f"{col} (returns)"))

adf_results = pd.DataFrame(tests).set_index("Series")
print("ADF Stationarity Tests")
print("Null hypothesis: series has a unit root (non-stationary)\n")
print(adf_results)

adf_results.to_csv("../data/table_2_adf.csv")

## Correlation Matrix

The correlation matrix gives a first look at how the series co-move within a trading day. I include the VIX change (not the level, which is too persistent for meaningful correlations) alongside the four CEE return series.

First, the VIX change should correlate negatively with all four CEE return series. If rising US fear coincides with falling CEE returns, the predictive analysis in later notebooks has a good foundation. Second, the four CEE markets should correlate positively with each other, since they're all exposed to the European business cycle. Heterogeneity in those pairwise correlations is informative to show how integrated each market is with its neighbours.

In [ ]:
# correlation matrix among VIX change and the four CEE return series
# using VIX change
corr_data = pd.concat([
    vix["VIX"].diff().rename("VIX_change"),
    cee_returns
], axis=1).dropna()

corr_matrix = corr_data.corr().round(3)

print("Correlation Matrix")
print("(contemporaneous, daily, 2018-2024)\n")
print(corr_matrix)

corr_matrix.to_csv("../data/table_3_correlations.csv")


Correlation Matrix
(contemporaneous, daily, 2018-2024)

            VIX_change  WIG20    BUX     PX    ATX
VIX_change       1.000 -0.354 -0.346 -0.343 -0.449
WIG20           -0.354  1.000  0.550  0.504  0.613
BUX             -0.346  0.550  1.000  0.551  0.589
PX              -0.343  0.504  0.551  1.000  0.705
ATX             -0.449  0.613  0.589  0.705  1.000

Saved → data/table_3_correlations.csv


## Correlation Heatmap

A 5×5 correlation table is hard to analyz3 visually, so using a heatmap would be optimal for display: blue for the negative VIX–CEE correlations, red shades for the positive CEE–CEE correlations, with darker colours indicating strenght of the links.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# mask the upper triangle for cleaner look
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    center=0,
    vmin=-0.6, vmax=1.0,
    square=True,
    cbar_kws={"shrink": 0.7, "label": "Correlation"},
    linewidths=0.5,
    linecolor="white",
    annot_kws={"size": 11},
    ax=ax
)
ax.set_title("Correlation matrix: VIX change and CEE log returns\n2018 to 2024, daily",
             fontsize=12, fontweight="500", pad=15)
plt.tight_layout()
plt.savefig("../figures/02_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Distribution Diagnostics

The final diagnostic check is visual: a histogram of each CEE return series with a normal distribution overlaid for comparison. The point is to make the non-normality of returns visible, not to formally test for it.

What I expect to see: a sharper peak at zero than the normal curve, and visibly fatter tails on both sides (especially the left). These are the empirical features that justify using HAC standard errors in the regressions, and the histogram is a more intuitive demonstration of that than a Jarque-Bera test.

In [ ]:
# visualize the distribution of returns to see fat tails and skewness
# this confirms why the descriptive stats look the way they do

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

colors = {"WIG20": "#2563eb", "BUX": "#7c3aed",
          "PX": "#059669", "ATX": "#d97706"}

for i, col in enumerate(["WIG20", "BUX", "PX", "ATX"]):
    series = cee_returns[col].dropna() * 100  # convert to percent
    
    # histogram
    axes[i].hist(series, bins=60, color=colors[col],
                 alpha=0.7, edgecolor="white", linewidth=0.3)
    
    # overlay normal distribution for comparison
    x = np.linspace(series.min(), series.max(), 200)
    mu, sigma = series.mean(), series.std()
    normal_pdf = stats.norm.pdf(x, mu, sigma)
    # scale normal PDF to histogram height
    bin_width = (series.max() - series.min()) / 60
    axes[i].plot(x, normal_pdf * len(series) * bin_width,
                 color="black", linewidth=1.2, linestyle="--",
                 label="Normal")
    
    axes[i].set_title(f"{col} daily returns (%)", fontsize=11, fontweight="500")
    axes[i].set_xlabel("Daily return (%)", fontsize=9)
    axes[i].legend(fontsize=9, loc="upper left")
    axes[i].tick_params(labelsize=9)

fig.suptitle("Return distributions vs Normal: visible fat tails in all four markets",
             fontsize=13, fontweight="500", y=1.01)
plt.tight_layout()
plt.savefig("../figures/03_return_distributions.png", dpi=150, bbox_inches="tight")
plt.show()